# Módulo 3 - Ejercicio 3 (Solución Completa)
## Transpilación a Hardware Real, Simulación Ruidosa y Ventaja GPU

Este notebook contiene la solución integral para el análisis de transpilación bajo topología restringida, la inclusión de modelos de ruido realistas, y la medición en tiempo real de la aceleración por GPU (NVIDIA A100) frente a un solo núcleo de CPU.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

### ️ Tarea 1: Transpilación con Topología Lineal
Construimos el circuito de Grover de 3 qubits y lo compilamos en un mapa lineal ($0 \leftrightarrow 1 \leftrightarrow 2$) comparando los niveles de optimización 0 y 3.


In [ ]:
# Construcción generalizada del circuito de Grover
def construir_grover(n_qubits: int, objetivo: str = None, iteraciones: int = 1) -> QuantumCircuit:
    """Construye el circuito del Algoritmo de Grover para n cúbits.
    
    Parámetros:
    - n_qubits: Número total de cúbits.
    - objetivo: Cadena binaria del estado solución (ej: '110'). Si es None, busca |1...1>.
    - iteraciones: Número de iteraciones de oráculo + difusor.
    """
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    
    if objetivo is None:
        objetivo = '1' * n_qubits
        
    ultimo = n_qubits - 1
    controles = list(range(ultimo))
    
    for _ in range(iteraciones):
        # Oráculo: envolvente X en los bits '0' (convención Little-Endian)
        for q, bit in enumerate(reversed(objetivo)):
            if bit == '0':
                qc.x(q)
                
        # Inversión de fase condicional
        qc.h(ultimo)
        qc.mcx(controles, ultimo)
        qc.h(ultimo)
        
        for q, bit in enumerate(reversed(objetivo)):
            if bit == '0':
                qc.x(q)
                
        # Difusor de n cúbits
        qc.h(range(n_qubits))
        qc.x(range(n_qubits))
        qc.h(ultimo)
        qc.mcx(controles, ultimo)
        qc.h(ultimo)
        qc.x(range(n_qubits))
        qc.h(range(n_qubits))
        
    qc.measure_all()
    return qc

# Circuito de Grover de 3 cúbits para |110> con 2 iteraciones óptimas
qc_base = construir_grover(3, objetivo="110", iteraciones=2)
mapa_lineal = CouplingMap([[0, 1], [1, 0], [1, 2], [2, 1]])
compuertas_base = ['cx', 'id', 'rz', 'sx', 'x']

t_opt0 = transpile(qc_base, coupling_map=mapa_lineal, basis_gates=compuertas_base, optimization_level=0, seed_transpiler=42)
t_opt3 = transpile(qc_base, coupling_map=mapa_lineal, basis_gates=compuertas_base, optimization_level=3, seed_transpiler=42)

print("--- Resultados de Transpilación Lineal ---")
print(f"Optimization Level 0 -> Profundidad: {t_opt0.depth()}, Compuertas CX: {t_opt0.count_ops().get('cx', 0)}")
print(f"Optimization Level 3 -> Profundidad: {t_opt3.depth()}, Compuertas CX: {t_opt3.count_ops().get('cx', 0)}")


### ️ Tarea 2: Creación de Modelo de Ruido y Simulación


In [ ]:
# Definición del modelo de ruido y simulación ruidosa
nm = NoiseModel()
err_2q = depolarizing_error(0.020, 2)
nm.add_all_qubit_quantum_error(err_2q, ["cx"])

backend_ruidoso = AerSimulator(noise_model=nm)
job = backend_ruidoso.run(t_opt3, shots=2048)
counts_ruidoso = job.result().get_counts()

prob_exito = counts_ruidoso.get("110", 0) / 2048 * 100
print("--- Resultados de Simulación Ruidosa ---")
print(f"Conteos obtenidos: {counts_ruidoso}")
print(f"Probabilidad de éxito ruidosa: {prob_exito:.2f}% (frente al ~94% en simulación ideal)")


### ️ Tarea 3 (Conexión HPC): Medición en Vivo de CPU (1 núcleo) vs. GPU NVIDIA A100 en el Clúster

En esta sección medimos en tiempo real el rendimiento del sistema evaluando el mismo circuito en:
1. **CPU Clásico Monohilo (1 solo núcleo, sin OpenMP):** `AerSimulator(method="statevector", device="CPU", max_parallel_threads=1)`
2. **GPU NVIDIA A100:** `AerSimulator(method="statevector", device="GPU")`

> **Nota técnica sobre el entorno GPU:** El paquete estándar `qiskit-aer` viene compilado únicamente para CPU. Para habilitar la GPU NVIDIA A100 en el clúster, se debe instalar el paquete con binarios CUDA correspondientes mediante:
> `pip install qiskit-aer-gpu-cu11` (y reiniciar el kernel de Jupyter).

In [ ]:
# Configuración de los simuladores para la comparativa
# CPU: 1 solo núcleo (sin paralelismo OpenMP)
sim_cpu1 = AerSimulator(method="statevector", device="CPU", max_parallel_threads=1)

# GPU NVIDIA A100:
try:
    sim_gpu = AerSimulator(method="statevector", device="GPU")
    # Verificación de ejecución real en el backend C++/CUDA
    sim_gpu.run(QuantumCircuit(2)).result()
    gpu_disponible = True
    print(" GPU NVIDIA detectada y activa para mediciones en vivo.")
except Exception as e:
    gpu_disponible = False
    print(f"Aviso: GPU NVIDIA no activa en este entorno ({e}).")
    print("Para habilitar la GPU con NVIDIA A100:")
    print("  1. Instala el paquete con soporte CUDA: !pip install qiskit-aer-gpu-cu11")
    print("  2. Reinicia el Kernel de Jupyter (Kernel -> Restart Kernel)")
    print("  3. Verifica con AerSimulator().available_devices() que aparezca ('CPU', 'GPU')")

In [ ]:
# Medición en tiempo real sobre el sistema: Algoritmo de Grover (CPU 1 núcleo vs GPU A100)
qubits_eval = [16, 18, 20, 22, 24, 26]
tiempos_cpu = []
tiempos_gpu = []

print(f"{'Qubits':<8}{'Amplitudes':<15}{'Tiempo CPU (1 núcleo)':<24}{'Tiempo GPU A100':<20}{'Aceleración':<10}")
print("=" * 77)

for n in qubits_eval:
    # Reutilizamos la función construir_grover para evaluar el algoritmo completo a escala n
    qc = construir_grover(n, iteraciones=1)
    
    # Medición en CPU (1 solo núcleo)
    t0 = time.time()
    sim_cpu1.run(qc, shots=500).result()
    t_cpu = time.time() - t0
    tiempos_cpu.append(t_cpu)
    
    # Medición en GPU NVIDIA
    if gpu_disponible:
        t0 = time.time()
        sim_gpu.run(qc, shots=500).result()
        t_gpu = time.time() - t0
        tiempos_gpu.append(t_gpu)
        sp = t_cpu / t_gpu
        sp_str = f"{sp:5.1f}x"
        t_gpu_str = f"{t_gpu:6.3f} s"
    else:
        t_gpu_str = "No disponible"
        sp_str = "N/A"
        
    print(f"{n:<8}{2**n:<15,d}{t_cpu:6.3f} s{' '*16}{t_gpu_str:<20}{sp_str:<10}")

In [ ]:
# Visualización con Matplotlib utilizando exclusivamente los datos medidos en vivo
if gpu_disponible and len(tiempos_gpu) == len(tiempos_cpu):
    speedup = np.array(tiempos_cpu) / np.array(tiempos_gpu)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=100)

    # Panel 1: Tiempos de Ejecución medidos (escala logarítmica)
    ax1.plot(qubits_eval, tiempos_cpu, "o--", color="#dc2626", linewidth=2, markersize=8, label="CPU (1 solo núcleo, sin OpenMP)")
    ax1.plot(qubits_eval, tiempos_gpu, "s-", color="#16a34a", linewidth=2.5, markersize=8, label="GPU NVIDIA A100 (cuQuantum)")
    ax1.set_yscale("log")
    ax1.set_xlabel("Número de Qubits ($n$)", fontsize=11, fontweight="bold")
    ax1.set_ylabel("Tiempo de Simulación (segundos, escala log)", fontsize=11, fontweight="bold")
    ax1.set_title("Escalamiento Real del Sistema: CPU (1 núcleo) vs. GPU A100", fontsize=12, fontweight="bold")
    ax1.set_xticks(qubits_eval)
    ax1.grid(True, which="both", linestyle=":", alpha=0.6)
    ax1.legend(frameon=True, facecolor="#f8fafc", edgecolor="#cbd5e1")

    # Panel 2: Factor de aceleración medido
    bars = ax2.bar(qubits_eval, speedup, width=1.2, color="#2563eb", edgecolor="#1e40af", alpha=0.85, label="Aceleración real medida")
    for bar, s in zip(bars, speedup):
        yval = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2.0, yval + (max(speedup)*0.02), f"{s:.1f}x", ha="center", va="bottom", fontweight="bold", color="#1e3a8a")

    ax2.set_xlabel("Número de Qubits ($n$)", fontsize=11, fontweight="bold")
    ax2.set_ylabel("Factor de Aceleración ($t_{\\mathrm{CPU}} / t_{\\mathrm{GPU}}$)", fontsize=11, fontweight="bold")
    ax2.set_title("Factor de Aceleración Medido en el Sistema", fontsize=12, fontweight="bold")
    ax2.set_xticks(qubits_eval)
    ax2.set_ylim(0, max(speedup) * 1.18)
    ax2.grid(True, linestyle=":", alpha=0.6, axis="y")
    ax2.legend(frameon=True, facecolor="#f8fafc", edgecolor="#cbd5e1")

    plt.tight_layout()
    plt.show()
else:
    # Si se corrió en un entorno sin GPU, graficar los tiempos reales medidos en CPU
    plt.figure(figsize=(7, 4.5), dpi=100)
    plt.plot(qubits_eval, tiempos_cpu, "o--", color="#dc2626", linewidth=2, markersize=8, label="CPU (1 solo núcleo, sin OpenMP)")
    plt.yscale("log")
    plt.xlabel("Número de Qubits ($n$)", fontsize=11, fontweight="bold")
    plt.ylabel("Tiempo de Simulación (segundos, escala log)", fontsize=11, fontweight="bold")
    plt.title("Escalamiento Real Medido en CPU (1 núcleo)", fontsize=12, fontweight="bold")
    plt.xticks(qubits_eval)
    plt.grid(True, which="both", linestyle=":", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("Aviso: Ejecuta este notebook en el clúster con GPU A100 para generar la comparativa completa.")